# Análise dos evals de tags MOPEP

Este notebook consolida os resultados salvos em `analytics/results/mopep-tags` e compara qualidade, latência e tamanho dos modelos. Ele não executa inferências; gere os runs primeiro com `make eval-run`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'analytics').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from analytics.src.consolidate_mopep_evals import collect_runs

sns.set_theme(style='whitegrid')
RESULTS_ROOT = PROJECT_ROOT / 'analytics/results/mopep-tags'
runs_df, predictions_df = collect_runs(RESULTS_ROOT)
if runs_df.empty:
    raise ValueError(f'Nenhum eval completo encontrado em {RESULTS_ROOT}')

print(f'{len(runs_df)} runs e {len(predictions_df)} predições carregados')

## Grupos comparáveis

Um grupo reúne somente runs com o mesmo dataset, exemplos, prompt e opções do Ollama. Por padrão, selecionamos o grupo com mais modelos. Para analisar outro grupo, substitua `COMPARISON_ID` por um valor da tabela.

In [ ]:
comparison_groups = (
    runs_df.groupby(['comparison_id', 'dataset_name', 'dataset_rows'], as_index=False)
    .agg(
        runs=('run_id', 'nunique'),
        models=('model_label', 'nunique'),
        first_run=('created_at', 'min'),
        last_run=('created_at', 'max'),
    )
    .sort_values(['models', 'runs'], ascending=False)
)
display(comparison_groups)

COMPARISON_ID = comparison_groups.iloc[0]['comparison_id']
selected_runs = runs_df.loc[runs_df['comparison_id'].eq(COMPARISON_ID)].copy()
selected_predictions = predictions_df.loc[
    predictions_df['comparison_id'].eq(COMPARISON_ID)
].copy()

## Resumo por modelo

In [ ]:
model_summary = (
    selected_runs.groupby('model_label', as_index=False)
    .agg(
        runs=('run_id', 'nunique'),
        requests=('requests', 'sum'),
        macro_f1=('macro_f1', 'mean'),
        mean_individual_f1=('mean_individual_f1', 'mean'),
        exact_match_rate=('exact_match_rate', 'mean'),
        parse_rate=('parse_rate', 'mean'),
        mean_latency_ms=('mean_latency_ms', 'mean'),
        parameter_size_b=('parameter_size_b', 'first'),
        quantization_level=('quantization_level', 'first'),
        model_size_gb=('model_size_gb', 'first'),
    )
    .sort_values('macro_f1', ascending=False)
)
display(model_summary)

## Qualidade agregada e distribuição por exemplo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, max(4, len(model_summary) * 0.7)))
sns.barplot(data=model_summary, y='model_label', x='macro_f1', ax=axes[0], color='steelblue')
axes[0].set(title='Macro F1 por modelo', xlabel='Macro F1', ylabel='Modelo', xlim=(0, 1))

sns.boxplot(data=selected_predictions, y='model_label', x='f1', ax=axes[1], color='lightsteelblue')
axes[1].set(title='Distribuição do F1 por exemplo', xlabel='F1 individual', ylabel='Modelo', xlim=(0, 1))
plt.tight_layout()
plt.show()

## Trade-off entre qualidade e latência

Modelos mais próximos do canto superior esquerdo combinam maior Macro F1 com menor latência média.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=model_summary,
    x='mean_latency_ms',
    y='macro_f1',
    hue='quantization_level',
    size='parameter_size_b',
    sizes=(100, 350),
)
for row in model_summary.itertuples():
    plt.annotate(row.model_label, (row.mean_latency_ms, row.macro_f1), xytext=(6, 5), textcoords='offset points')
plt.title('Qualidade versus latência por modelo')
plt.xlabel('Latência média por requisição (ms)')
plt.ylabel('Macro F1')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Qualidade versus tamanho e quantização

Runs novos registram o tamanho informado pelo Ollama. Quando ele não está disponível, o gráfico usa o número de parâmetros identificado no nome do modelo.

In [ ]:
if model_summary['model_size_gb'].notna().any():
    size_column = 'model_size_gb'
    size_label = 'Tamanho armazenado no Ollama (GB)'
else:
    size_column = 'parameter_size_b'
    size_label = 'Parâmetros do modelo (bilhões)'

compression_plot = model_summary.dropna(subset=[size_column])
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=compression_plot,
    x=size_column,
    y='macro_f1',
    hue='quantization_level',
    s=180,
)
for row in compression_plot.itertuples():
    plt.annotate(row.model_label, (getattr(row, size_column), row.macro_f1), xytext=(6, 5), textcoords='offset points')
plt.title('Qualidade versus tamanho e quantização')
plt.xlabel(size_label)
plt.ylabel('Macro F1')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Exemplos com menor F1

In [ ]:
error_columns = [
    column for column in [
        'example_id', 'domain', 'model_label', 'expected_tags', 'generated_tags',
        'missing_tags', 'extra_tags', 'f1', 'latency_ms', 'parse_ok',
    ]
    if column in selected_predictions.columns
]
worst_examples = selected_predictions.sort_values(['f1', 'latency_ms']).head(20)
display(worst_examples[error_columns])